# Measuring in other bases
**Quantum Computing · Week 3 (The maths of qubits, and interference) · Day 4**

Lesson page: https://sara-aminpour.com/tutorials/qc/w3d4.html

In [ ]:
# On Google Colab, this installs qiskit (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2

A meter only reads 0 or 1. To ask "is it $|+\rangle$ or $|-\rangle$?", turn the state first:
H before the meter measures in the **X basis**; S† then H measures in the **Y basis**.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.primitives import StatevectorSampler
from qiskit.quantum_info import Statevector

sampler = StatevectorSampler(seed=3)

def prepare(label):
    qc = QuantumCircuit(1)
    if label == "+":
        qc.h(0)
    elif label == "-":
        qc.x(0)
        qc.h(0)
    return qc

for label in ["+", "-"]:
    z = prepare(label)
    z.measure_all()
    x = prepare(label)
    x.h(0)                      # turn the x axis onto the z axis...
    x.measure_all()             # ...then use the ordinary meter
    rz, rx = sampler.run([z, x], shots=1000).result()
    print(f"|{label}>:  Z basis {rz.data.meas.get_counts()}   X basis {rx.data.meas.get_counts()}  (0 means '+', 1 means '-')")

## Any state, any basis
`measure_in` appends the basis change and a measurement. For the Y basis: S† (`sdg`) then H.

In [ ]:
def measure_in(state_circuit, basis, shots=4000):
    qc = state_circuit.copy()
    if basis == "X":
        qc.h(0)
    elif basis == "Y":
        qc.sdg(0)
        qc.h(0)
    qc.measure_all()
    counts = sampler.run([qc], shots=shots).result()[0].data.meas.get_counts()
    return counts.get("0", 0) / shots           # probability of the first outcome (0, + or +i)

plus_i = QuantumCircuit(1)
plus_i.h(0)
plus_i.s(0)                                     # |+i>
for basis in "ZXY":
    print(f"|+i> in the {basis} basis: P(first outcome) = {measure_in(plus_i, basis):.3f}")

## Each basis reads one coordinate of the arrow
$P(\text{first outcome}) = (1 + \text{coordinate})/2$, so coordinate $= 2P - 1$. Compare with the exact Bloch vector.

In [ ]:
state = QuantumCircuit(1)
state.ry(1.1, 0)
state.rz(0.7, 0)
measured = [2 * measure_in(state, b) - 1 for b in "XYZ"]
a, b = Statevector(state).data
c = np.conj(a) * b
exact = [2 * c.real, 2 * c.imag, abs(a) ** 2 - abs(b) ** 2]
print("from three measurements:", np.round(measured, 3))
print("exact Bloch vector:     ", np.round(exact, 3))

## Your turn: predict, then measure
For $R_Y(60°)|0\rangle$ the Bloch vector is $(\sin 60°, 0, \cos 60°) \approx (0.866, 0, 0.5)$.
Predict $P(0)$, $P(+)$ and $P(+i)$ with $(1 + \text{coordinate})/2$, then measure them.

In [ ]:
ry60 = QuantumCircuit(1)
ry60.ry(np.radians(60), 0)
predicted = {"Z": None, "X": None, "Y": None}    # TODO: your three predictions
for basis in "ZXY":
    print(f"{basis}: predicted {predicted[basis]}, measured {measure_in(ry60, basis):.3f}")

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
predicted = {"Z": (1 + 0.5) / 2, "X": (1 + np.sin(np.radians(60))) / 2, "Y": (1 + 0) / 2}
for basis in "ZXY":
    print(f"{basis}: predicted {predicted[basis]:.3f}, measured {measure_in(ry60, basis):.3f}")
print("The arrow leans toward +x, so the X meter says '+' most often, and it has no y component: 50/50 in Y.")